In [ ]:
# Define LSTM Model
import random
import os
import numpy as np
import tensorflow as tf
from tensorflow.keras import Input
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout, Input
from sklearn.metrics import mean_squared_error, mean_absolute_error

def build_lstm_model(input_shape):
    model = Sequential([
        Input(shape=input_shape),
        LSTM(32),
        Dense(16, activation='relu'),
        Dropout(0.1),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse')
    return model
    
# Setup
data_dir = "../windowed_data/6h"
feature_names = ['bg', 'cals', 'dist', 'hr', 'insulin', 'steps']
selected_features = ['bg', 'steps', 'dist', 'insulin']
selected_indices = [feature_names.index(f) for f in selected_features]
all_files = [f for f in os.listdir(data_dir) if f.endswith('.npz')]

results_lstm = {}
all_preds_lstm = {}

# Train & Evaluate per Patient
for file in all_files:
    pid = file.replace(".npz", "")
    data = np.load(os.path.join(data_dir, file))
    X, y = data['X'], data['y']
    X = X[:, :, selected_indices]

    # Chronological 80/20 split
    split = int(0.8 * len(X))
    X_train, X_test = X[:split], X[split:]
    y_train, y_test = y[:split], y[split:]

    print(f"\nTraining LSTM for {pid}...")

    model = build_lstm_model(input_shape=(X.shape[1], X.shape[2]))
    model.fit(X_train, y_train, epochs=10, batch_size=32, verbose=1)

    y_pred = model.predict(X_test).flatten()
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)

    results_lstm[pid] = {"rmse": rmse, "mae": mae}
    all_preds_lstm[pid] = {"y_true": y_test.tolist(), "y_pred": y_pred.tolist()}  # For CEG

    print(f"{pid} - RMSE: {rmse:.3f} | MAE: {mae:.3f}")


Training LSTM for P01...
Epoch 1/10
1457/1457 ━━━━━━━━━━━━━━━━━━━━ 78s 52ms/step - loss: 23.3219
Epoch 2/10
1457/1457 ━━━━━━━━━━━━━━━━━━━━ 98s 67ms/step - loss: 7.5162
Epoch 3/10
1457/1457 ━━━━━━━━━━━━━━━━━━━━ 119s 51ms/step - loss: 7.3133
Epoch 4/10
1457/1457 ━━━━━━━━━━━━━━━━━━━━ 62s 42ms/step - loss: 7.0016
Epoch 5/10
1457/1457 ━━━━━━━━━━━━━━━━━━━━ 75s 38ms/step - loss: 6.9295
Epoch 6/10
1457/1457 ━━━━━━━━━━━━━━━━━━━━ 81s 37ms/step - loss: 6.8290
Epoch 7/10
1457/1457 ━━━━━━━━━━━━━━━━━━━━ 49s 33ms/step - loss: 6.5657
Epoch 8/10
1457/1457 ━━━━━━━━━━━━━━━━━━━━ 91s 40ms/step - loss: 6.5904
Epoch 9/10
1457/1457 ━━━━━━━━━━━━━━━━━━━━ 46s 31ms/step - loss: 6.3431
Epoch 10/10
1457/1457 ━━━━━━━━━━━━━━━━━━━━ 44s 30ms/step - loss: 6.2778
365/365 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step
P01 - RMSE: 2.511 | MAE: 1.874

Training LSTM for P02...
Epoch 1/10
1506/1506 ━━━━━━━━━━━━━━━━━━━━ 63s 35ms/step - loss: 96.5019
Epoch 2/10
1506/1506 ━━━━━━━━━━━━━━━━━━━━ 94s 43ms/step - loss: 71.9497
Epoch 3/10
1506/15

In [ ]:
import numpy as np
import pandas as pd

# Exporting and display the results
df_results_lstm = pd.DataFrame(results_lstm).T.round(3)
df_results_lstm.index.name = "Patient ID"

# Results visualisation
import matplotlib.pyplot as plt

# RMSE Plot
df_rmse_sorted_lstm = df_results_lstm.sort_values("rmse")
x_rmse_lstm = np.arange(len(df_rmse_sorted_lstm))
rmse_values_lstm = df_rmse_sorted_lstm["rmse"].values
mean_rmse_lstm = rmse_values_lstm.mean()
std_rmse_lstm = rmse_values_lstm.std()

plt.figure(figsize=(12, 5))
plt.bar(x_rmse_lstm, rmse_values_lstm, color='skyblue')
plt.xticks(x_rmse_lstm, df_rmse_sorted_lstm.index, rotation=45)

# Confidence band 
plt.axhspan(mean_rmse_lstm - std_rmse_lstm, mean_rmse_lstm + std_rmse_lstm, color='red', alpha=0.2, label="±1 SD")
plt.axhline(mean_rmse_lstm, color='red', linestyle='--', label=f"Mean RMSE = {mean_rmse_lstm:.2f}")

plt.title("RMSE per Patient - LSTM")
plt.ylabel("RMSE")
plt.legend()
plt.tight_layout()
plt.show()


# MAE Plot
df_mae_sorted_lstm = df_results_lstm.sort_values("mae")
x_mae_lstm = np.arange(len(df_mae_sorted_lstm))
mae_values_lstm = df_mae_sorted_lstm["mae"].values
mean_mae_lstm = mae_values_lstm.mean()
std_mae_lstm = mae_values_lstm.std()

plt.figure(figsize=(12, 5))
plt.bar(x_mae_lstm, mae_values_lstm, color='skyblue')
plt.xticks(x_mae_lstm, df_mae_sorted_lstm.index, rotation=45)

# Confidence band 
plt.axhspan(mean_mae_lstm - std_mae_lstm, mean_mae_lstm + std_mae_lstm, color='green', alpha=0.2, label="±1 SD")
plt.axhline(mean_mae_lstm, color='green', linestyle='--', label=f"Mean MAE = {mean_mae_lstm:.2f}")

plt.title("MAE per Patient - LSTM")
plt.ylabel("MAE")
plt.legend()
plt.tight_layout()
plt.show()

In [3]:
df_results_lstm.to_csv("../output_baseline/lstm_per_patient_results.csv")

with open("../output_baseline/lstm_predictions_per_patient.json", "w") as f:
    json.dump(all_preds_lstm, f)

# Performance Analysis
print("Average RMSE:", df_results_lstm['rmse'].mean())
print("Average MAE:", df_results_lstm['mae'].mean())

OSError: Cannot save file into a non-existent directory: '..\output_baseline'

In [ ]:
import os
import json
import pandas as pd

# Load saved predictions
with open("../output_baseline/lstm_predictions_per_patient.json", "r") as f:
    all_preds_lstm = json.load(f)

# Convert to DataFrame
rows = []
for pid, data in all_preds_lstm.items():
    for true_val, pred_val in zip(data["y_true"], data["y_pred"]):
        rows.append({
            "patient_id": pid,
            "true_bg": float(true_val),
            "predicted_bg": float(pred_val)
        })

df_lstm = pd.DataFrame(rows)

# Convert mmol/L to mg/dL
df_lstm["true_bg_mg"] = df_lstm["true_bg"] * 18
df_lstm["predicted_bg_mg"] = df_lstm["predicted_bg"] * 18

# Clarke Error Grid Zone Classification (using mg/dL)
def classify_ceg_zone(true, pred):
    true = float(true)
    pred = float(pred)

    # Zone A: within 20% of reference BG or both < 70
    if (abs(true - pred) <= 20) or (true < 70 and pred < 70) or (true > 180 and pred > 180 and abs(true - pred) <= 20):
        return 'A'
    
    # Zone B: outside 20% but no treatment error (e.g. still within euglycemic range)
    if (true >= 70 and true <= 180 and pred >= 70 and pred <= 180):
        return 'B'
    if (true < 70 and 70 <= pred <= 180):
        return 'B'
    if (true > 180 and 70 <= pred <= 180):
        return 'B'
    
    # Zone E: dangerous confusion
    if (true < 70 and pred > 180) or (true > 180 and pred < 70):
        return 'E'
    
    # Zone D: dangerous failure to detect and treat
    if (true >= 180 and pred <= 180) or (true < 70 and pred >= 70):
        return 'D'
    
    # Zone C: overcorrection / unnecessary treatment
    return 'C'

# Apply Classification
df_lstm["zone"] = df_lstm.apply(lambda row: classify_ceg_zone(row["true_bg_mg"], row["predicted_bg_mg"]), axis=1)

# Summary Table Per Patient
ceg_summary_lstm = df_lstm.groupby(['patient_id', 'zone']).size().unstack(fill_value=0)
ceg_summary_lstm['Total'] = ceg_summary_lstm.sum(axis=1)
ceg_summary_percent_lstm = ceg_summary_lstm.div(ceg_summary_lstm['Total'], axis=0) * 100
ceg_summary_percent_lstm = ceg_summary_percent_lstm.round(2)

# Save & Display
ceg_summary_percent_lstm.to_csv("../output_baseline/ceg_zone_summary_per_patient_lstm.csv")
print("\nClarke Error Grid Zone Summary (in %):")
display(ceg_summary_percent_lstm)

In [ ]:
# Patient-Level Clarke Error Grid Bar Plot

ceg_plot_lstm = ceg_summary_percent_lstm.drop(columns="Total").fillna(0)

plt.figure(figsize=(12, 6))
ceg_plot_lstm.plot(kind="bar", stacked=True, colormap="tab20", figsize=(14, 6))
plt.ylabel("Percentage (%)")
plt.title("Clarke Error Grid Zone Distribution per Patient (LSTM)")
plt.legend(title="Zone", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.grid(axis='y')
plt.show()